In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
import tensorflow.keras.backend as K


IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 40
LEARNING_RATE = 0.0001
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

BRISC_IMG_DIR = "/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/segmentation_task/train/images"
BRISC_MASK_DIR = "/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/segmentation_task/train/masks"
MIXED_BASE_DIR = "/kaggle/input/datasets/wajeehaldeenaljunaid/mixed-brain-tumor-dataset/mixed_brain_tumor_dataset"

In [3]:
all_data_pairs = []

# BRISC Dataset
brisc_count = 0
if os.path.exists(BRISC_IMG_DIR):
    img_files = [f for f in os.listdir(BRISC_IMG_DIR) if f.endswith('.jpg')]
    for f in sorted(img_files):
        img_path = os.path.join(BRISC_IMG_DIR, f)
        mask_path = os.path.join(BRISC_MASK_DIR, f.replace('.jpg', '.png'))
        if os.path.exists(mask_path):
            all_data_pairs.append((img_path, mask_path))
            brisc_count += 1

# Mixed Dataset
mixed_count = 0
if os.path.exists(MIXED_BASE_DIR):
    
    for cls in ['0', '1', '2', '3']:
        img_dir = os.path.join(MIXED_BASE_DIR, 'image', cls)
        mask_dir = os.path.join(MIXED_BASE_DIR, 'mask', cls)
        
        if os.path.exists(img_dir):
            cls_count = 0
            img_files = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
            
            for f in sorted(img_files):
                img_path = os.path.join(img_dir, f)
                mask_name = f.replace('.jpg', '_m.jpg')
                mask_path = os.path.join(mask_dir, mask_name)
                if os.path.exists(mask_path):
                    all_data_pairs.append((img_path, mask_path))
                    mixed_count += 1
                    cls_count += 1
    
print(f"TOTAL: {len(all_data_pairs)} image-mask pairs")


print("Splitting")

train_val_pairs, test_pairs = train_test_split(all_data_pairs, test_size=TEST_SPLIT, random_state=42, shuffle=True)

val_size_adjusted = VAL_SPLIT / (TRAIN_SPLIT + VAL_SPLIT)
train_pairs, val_pairs = train_test_split(train_val_pairs, test_size=val_size_adjusted, random_state=42, shuffle=True)


class DataGenerator(keras.utils.Sequence):
    def __init__(self, data_pairs, batch_size, augment=False):
        self.data_pairs = data_pairs
        self.batch_size = batch_size
        self.augment = augment
    
    def __len__(self):
        return max(1, len(self.data_pairs) // self.batch_size)
    
    def __getitem__(self, idx):
        batch = self.data_pairs[idx * self.batch_size:(idx + 1) * self.batch_size]
        images, masks = [], []
        
        for img_path, mask_path in batch:
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None or mask is None:
                continue
            
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
            
            if self.augment:
                if np.random.random() > 0.5:
                    img = np.fliplr(img)
                    mask = np.fliplr(mask)
                if np.random.random() > 0.5:
                    img = np.flipud(img)
                    mask = np.flipud(mask)
            
            img = img.astype(np.float32) / 255.0
            mask = (mask > 0).astype(np.float32)
            
            images.append(np.stack([img, img, img], axis=-1))
            masks.append(np.expand_dims(mask, axis=-1))
        
        return np.array(images), np.array(masks)

train_gen = DataGenerator(train_pairs, BATCH_SIZE, augment=True)
val_gen = DataGenerator(val_pairs, BATCH_SIZE, augment=False)
test_gen = DataGenerator(test_pairs, BATCH_SIZE, augment=False)

In [ ]:
def build_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    
    resnet = keras.applications.ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)
    
    # 5 Skip
    s1 = resnet.get_layer('conv1_relu').output
    s2 = resnet.get_layer('conv2_block3_out').output
    s3 = resnet.get_layer('conv3_block4_out').output
    s4 = resnet.get_layer('conv4_block6_out').output
    bridge = resnet.get_layer('conv5_block3_out').output
    
    def up_block(x, skip, filters):
        x = layers.UpSampling2D(2)(x)
        x = layers.Concatenate()([x, skip])
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        return x
    
    u1 = up_block(bridge, s4, 512)
    u2 = up_block(u1, s3, 256)
    u3 = up_block(u2, s2, 128)
    u4 = up_block(u3, s1, 64)
    
    up = layers.UpSampling2D(2)(u4)
    out = layers.Conv2D(1, 1, activation='sigmoid')(up)
    
    return models.Model(inputs, out, name='UNet_ResNet50')

print("Building model")
model = build_model()
print(f"Model built. Parameters: {model.count_params():,}\n")

In [ ]:
def tversky_loss(y_true, y_pred, alpha=0.3, beta=0.7, smooth=0.000001):
    y_true_flat = K.flatten(y_true)
    y_pred_flat = K.flatten(y_pred)
    tp = K.sum(y_true_flat * y_pred_flat)
    fp = K.sum((1 - y_true_flat) * y_pred_flat)
    fn = K.sum(y_true_flat * (1 - y_pred_flat))
    tversky = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
    return 1.0 - tversky

def dice_coef(y_true, y_pred, smooth=0.000001):
    y_true_flat = K.flatten(y_true)
    y_pred_flat = K.flatten(y_pred)
    intersection = K.sum(y_true_flat * y_pred_flat)
    union = K.sum(y_true_flat) + K.sum(y_pred_flat)
    return (2.0 * intersection + smooth) / (union + smooth)

def sensitivity(y_true, y_pred, threshold=0.5):
    y_pred_binary = K.cast(y_pred > threshold, 'float32')
    tp = K.sum(y_true * y_pred_binary)
    fn = K.sum(y_true * (1 - y_pred_binary))
    return tp / (tp + fn + K.epsilon())

def precision(y_true, y_pred, threshold=0.5):
    y_pred_binary = K.cast(y_pred > threshold, 'float32')
    tp = K.sum(y_true * y_pred_binary)
    fp = K.sum((1 - y_true) * y_pred_binary)
    return tp / (tp + fp + K.epsilon())

def iou_score(y_true, y_pred, threshold=0.5, smooth=000001):
    y_pred_binary = K.cast(y_pred > threshold, 'float32')
    intersection = K.sum(y_true * y_pred_binary)
    union = K.sum(y_true) + K.sum(y_pred_binary) - intersection
    return (intersection + smooth) / (union + smooth)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tversky_loss,
    metrics=[dice_coef, sensitivity, precision, iou_score]
)

print("Model compil")

In [ ]:
callbacks_list = [
    callbacks.ModelCheckpoint(
        'best_segmentation_model.keras',
        monitor='val_dice_coef', mode='max',
        save_best_only=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_dice_coef', mode='max',
        factor=0.5, patience=5, min_lr=0.0000001, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_dice_coef', mode='max',
        patience=15, restore_best_weights=True, verbose=1
    ),
    callbacks.CSVLogger('segmentation_log.csv', append=False)
]

In [ ]:
print("TRAINING")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    verbose=1
)

print("Done+++")

In [ ]:
test_results = model.evaluate(test_gen, verbose=0)
test_loss, test_dice, test_sensitivity, test_precision, test_iou = test_results

print("Test Results:")
print(f"  Dice Score  : {test_dice:.4f}")
print(f"  Sensitivity : {test_sensitivity:.4f}")
print(f"  Precision   : {test_precision:.4f}")
print(f"  IoU Score   : {test_iou:.4f}")
print(f"  Loss        : {test_loss:.4f}")

In [ ]:
fig = plt.figure(figsize=(20, 12))
epochs_range = range(1, len(history.history['dice_coef']) + 1)
best_epoch = np.argmax(history.history['val_dice_coef']) + 1

# 1. Dice Score
ax1 = plt.subplot(2, 3, 1)
ax1.plot(epochs_range, history.history['dice_coef'], 'o-', label='Train', markersize=3)
ax1.plot(epochs_range, history.history['val_dice_coef'], 's-', label='Val', markersize=3)
ax1.set_title('Dice Score', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Dice')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Loss
ax2 = plt.subplot(2, 3, 2)
ax2.plot(epochs_range, history.history['loss'], 'o-', label='Train', markersize=3)
ax2.plot(epochs_range, history.history['val_loss'], 's-', label='Val', markersize=3)
ax2.set_title('Tversky Loss', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Sensitivity
ax3 = plt.subplot(2, 3, 3)
ax3.plot(epochs_range, history.history['sensitivity'], 'o-', label='Train', markersize=3, color='green')
ax3.plot(epochs_range, history.history['val_sensitivity'], 's-', label='Val', markersize=3, color='darkgreen')
ax3.set_title('Sensitivity (Recall)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Sensitivity')
ax3.legend()
ax3.grid(alpha=0.3)

# 4. Precision
ax4 = plt.subplot(2, 3, 4)
ax4.plot(epochs_range, history.history['precision'], 'o-', label='Train', markersize=3, color='blue')
ax4.plot(epochs_range, history.history['val_precision'], 's-', label='Val', markersize=3, color='darkblue')
ax4.set_title('Precision', fontsize=12, fontweight='bold')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Precision')
ax4.legend()
ax4.grid(alpha=0.3)

# 5. IoU
ax5 = plt.subplot(2, 3, 5)
ax5.plot(epochs_range, history.history['iou_score'], 'o-', label='Train', markersize=3, color='orange')
ax5.plot(epochs_range, history.history['val_iou_score'], 's-', label='Val', markersize=3, color='darkorange')
ax5.set_title('IoU Score', fontsize=12, fontweight='bold')
ax5.set_xlabel('Epoch')
ax5.set_ylabel('IoU')
ax5.legend()
ax5.grid(alpha=0.3)

# 6. Train-Val Gap
ax6 = plt.subplot(2, 3, 6)
dice_gap = [t - v for t, v in zip(history.history['dice_coef'], history.history['val_dice_coef'])]
ax6.plot(epochs_range, dice_gap, 'o-', color='red', linewidth=2, markersize=3)
ax6.set_title('Train-Val Dice Gap', fontsize=12, fontweight='bold')
ax6.set_xlabel('Epoch')
ax6.set_ylabel('Gap')
ax6.legend()
ax6.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('segmentation_report.png', dpi=300, bbox_inches='tight')
plt.show()


best_val_dice = max(history.history['val_dice_coef'])

print("FINAL RESULTS")
print(f"\nBest Model (Epoch {best_epoch}):")
print(f"Val Dice: {best_val_dice:.4f}")

print(f"Test Set:")
print(f"Dice: {test_dice:.4f}")
print(f"Sensitivity: {test_sensitivity:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Iou: {test_iou:.4f}")

In [ ]:
model.save('final_segmentation_model.keras')
missed_pixels = (1 - test_sensitivity) * 100
print(f"Sensitivity: {test_sensitivity:.4f} (Missed: {missed_pixels:.2f}%)")